In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

GIORNI_BORSA = 252

px = yf.download(["ESGU", "SPY", "XLE", "XLK"],
                 start="2019-01-01", auto_adjust=True, progress=False)["Close"].dropna()

r = px.pct_change().dropna()
print(px.columns.tolist(), "|", len(r), "giorni |",
      r.index.min().date(), "→", r.index.max().date())
print("zeri per colonna:")
print((r == 0).sum())

['ESGU', 'SPY', 'XLE', 'XLK'] | 1888 giorni | 2019-01-03 → 2026-07-09
zeri per colonna:
Ticker
ESGU     8
SPY      2
XLE     11
XLK      3
dtype: int64


In [2]:
spread = r["ESGU"] - r["SPY"]           # cosa vogliamo spiegare
en_rel = r["XLE"] - r["SPY"]            # energia, al netto del mercato
tk_rel = r["XLK"] - r["SPY"]            # tech, al netto del mercato

X = np.column_stack([np.ones(len(r)), en_rel, tk_rel])
y = spread.values

beta, *_ = np.linalg.lstsq(X, y, rcond=None)
pred = X @ beta
r2 = 1 - ((y - pred)**2).sum() / ((y - y.mean())**2).sum()

print(f"intercetta:  {beta[0]*252*100:+.2f} pp/anno")
print(f"energia:     {beta[1]:+.4f}")
print(f"tech:        {beta[2]:+.4f}")
print(f"R²:          {r2:.3f}   (quota dello spread spiegata dai due settori)")

intercetta:  -0.38 pp/anno
energia:     -0.0015
tech:        +0.0350
R²:          0.057   (quota dello spread spiegata dai due settori)


In [3]:
for anno in range(2019, 2027):
    m = r.index.year == anno
    if m.sum() < 100: continue
    X = np.column_stack([np.ones(m.sum()), en_rel[m], tk_rel[m]])
    y = spread[m].values
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    p = X @ b
    r2 = 1 - ((y-p)**2).sum() / ((y-y.mean())**2).sum()
    print(f"{anno}:  alfa {b[0]*252*100:+6.2f} pp/a   en {b[1]:+.4f}   tk {b[2]:+.4f}   R² {r2:.3f}")

2019:  alfa  +0.91 pp/a   en +0.0138   tk +0.0112   R² 0.006
2020:  alfa  +2.11 pp/a   en +0.0014   tk +0.0818   R² 0.141
2021:  alfa  -1.71 pp/a   en -0.0005   tk +0.0592   R² 0.216
2022:  alfa  -1.85 pp/a   en -0.0002   tk +0.0568   R² 0.199
2023:  alfa  -0.61 pp/a   en -0.0039   tk +0.0120   R² 0.033
2024:  alfa  -0.44 pp/a   en -0.0025   tk +0.0197   R² 0.077
2025:  alfa  -0.87 pp/a   en -0.0071   tk +0.0059   R² 0.013
2026:  alfa  +0.22 pp/a   en +0.0023   tk +0.0170   R² 0.040


In [4]:
def valida(tickers, inizio="2019-01-01", soglia_zeri=0.02):
    """Quali ticker esistono, hanno storico, e non sono fermi?"""
    righe, scartati = [], []
    for t in tickers:
        try:
            d = yf.download(t, start=inizio, auto_adjust=True, progress=False)
            if isinstance(d.columns, pd.MultiIndex):
                d.columns = d.columns.droplevel("Ticker")
            s = d["Close"].dropna()
            if len(s) < 250:
                scartati.append((t, f"solo {len(s)} giorni")); continue
            zeri = int((s.pct_change().dropna() == 0).sum())
            righe.append({"ticker": t, "giorni": len(s), "zeri": zeri,
                          "%zeri": round(100*zeri/len(s), 1),
                          "da": s.index.min().date(),
                          "usabile": zeri/len(s) < soglia_zeri})
        except Exception as e:
            scartati.append((t, str(e)[:40]))
    return pd.DataFrame(righe), scartati

candidati = [
    "SWDA.MI",   # iShares Core MSCI World      — benchmark
    "XDWD.DE",   # Xtrackers MSCI World         — benchmark
    "SUSW.MI",   # iShares MSCI World SRI       — ESG severo
    "XZW0.DE",   # Xtrackers MSCI World ESG     — best-in-class
    "IWDA.AS",   # stesso di SWDA, altra borsa  — controllo
]

tab, ko = valida(candidati)
print("SCARTATI:", ko)
tab

SCARTATI: []


,ticker,giorni,zeri,%zeri,da,usabile
0,SWDA.MI,1909,19,1.0,2019-01-02,True
1,XDWD.DE,1910,4,0.2,2019-01-02,True
2,SUSW.MI,1793,69,3.8,2019-06-18,False
3,XZW0.DE,1911,22,1.2,2019-01-02,True
4,IWDA.AS,1925,13,0.7,2019-01-02,True


In [5]:
# 1. Verso Xetra, dove c'è liquidità. Aggiungo i due SRI/Screened.
universo = {
    "XDWD.DE": ("MSCI World",          "benchmark",  0.12),
    "EUNL.DE": ("MSCI World",          "benchmark",  0.20),
    "XZW0.DE": ("MSCI World ESG",      "esg",        0.20),
    "2B7K.DE": ("MSCI World SRI",      "esg",        0.20),
    "IQQW.DE": ("MSCI World Screened", "esg",        0.20),
}

tab_v, ko = valida(list(universo))
print("SCARTATI:", ko)
display(tab_v)

# 2. Solo i sopravvissuti, allineati e misurati
buoni = tab_v.loc[tab_v["usabile"], "ticker"].tolist()
res, px_u = confronta(buoni, inizio="2019-06-19")

# 3. Attacco le colonne che i prezzi non sanno
res["ter_%"]   = [universo[t][2] for t in res.index]
res["tipo"]    = [universo[t][1] for t in res.index]
res["indice"]  = [universo[t][0] for t in res.index]

print(res.attrs)
res

SCARTATI: []


,ticker,giorni,zeri,%zeri,da,usabile
0,XDWD.DE,1910,4,0.2,2019-01-02,True
1,EUNL.DE,1910,8,0.4,2019-01-02,True
2,XZW0.DE,1911,22,1.2,2019-01-02,True
3,2B7K.DE,1870,22,1.2,2019-02-27,True
4,IQQW.DE,1910,5,0.3,2019-01-02,True


NameError: name 'confronta' is not defined

In [ ]:
GIORNI_BORSA = 252

def confronta(tickers, inizio="2019-01-01", fine=None, rf_annuo=0.0):
    grezzi = yf.download(tickers, start=inizio, end=fine, auto_adjust=True, progress=False)
    px = grezzi["Close"].dropna()
    rend = px.pct_change().dropna()
    anni = len(rend) / GIORNI_BORSA
    rf_g = (1 + rf_annuo) ** (1 / GIORNI_BORSA) - 1
    righe = []
    for t in px.columns:
        r = rend[t]; curva = (1 + r).cumprod(); dd = curva / curva.cummax() - 1; ex = r - rf_g
        righe.append({
            "ticker": t,
            "cagr_%":   round(((px[t].iloc[-1] / px[t].iloc[0]) ** (1/anni) - 1) * 100, 2),
            "vol_%":    round(r.std() * np.sqrt(GIORNI_BORSA) * 100, 2),
            "sharpe":   round(ex.mean() / ex.std() * np.sqrt(GIORNI_BORSA), 2),
            "max_dd_%": round(dd.min() * 100, 2),
            "gg_sott_acqua": int((dd < -0.0001).sum()),
            "zeri":     int((r == 0).sum()),
        })
    tab = pd.DataFrame(righe).set_index("ticker")
    tab.attrs["periodo"] = f"{rend.index.min().date()} → {rend.index.max().date()}"
    tab.attrs["giorni"] = len(rend)
    tab.attrs["rf_%"] = rf_annuo * 100
    tab.attrs["fonte"] = "yahoo / Close auto_adjust"
    return tab, px

In [ ]:
buoni = tab_v.loc[tab_v["usabile"], "ticker"].tolist()
res, px_u = confronta(buoni, inizio="2019-02-28")

res["ter_%"]  = [universo[t][2] for t in res.index]
res["tipo"]   = [universo[t][1] for t in res.index]

# tracking error e correlazione contro il benchmark più liquido
r_u = px_u.pct_change().dropna()
bench = "XDWD.DE"
res["te_%"]   = [round((r_u[t] - r_u[bench]).std() * np.sqrt(252) * 100, 2) for t in res.index]
res["corr"]   = [round(r_u[t].corr(r_u[bench]), 4) for t in res.index]

print(res.attrs)
res.sort_values("sharpe", ascending=False)

In [ ]:
rng = np.random.default_rng(42)
n = len(r_u)
sh = lambda s: s.mean()/s.std()*np.sqrt(252)

print(f"{'ticker':10s} {'sharpe':>7s}   IC 90% bootstrap")
for t in res.index:
    d = np.array([sh(r_u[t].iloc[rng.integers(0,n,n)]) for _ in range(1000)])
    print(f"{t:10s} {sh(r_u[t]):7.2f}   [{np.percentile(d,5):.2f} ; {np.percentile(d,95):.2f}]")

In [ ]:
rng = np.random.default_rng(42)
n = len(r_u)
sh = lambda s: s.mean()/s.std()*np.sqrt(252)
bench = "XDWD.DE"

print(f"{'ticker':10s} {'ΔSharpe':>8s}   IC 90% (bootstrap appaiato)   zero dentro?")
for t in res.index:
    if t == bench: continue
    oss = sh(r_u[t]) - sh(r_u[bench])
    d = []
    for _ in range(2000):
        idx = rng.integers(0, n, n)          # righe intere: tutti gli ETF restano appaiati
        c = r_u.iloc[idx]
        d.append(sh(c[t]) - sh(c[bench]))
    lo, hi = np.percentile(d, [5, 95])
    print(f"{t:10s} {oss:+8.3f}   [{lo:+.3f} ; {hi:+.3f}]   {'SÌ' if lo<0<hi else 'NO'}")

In [ ]:
def punteggio(tab, pesi=None):
    """Normalizza ogni colonna 0-100, poi somma pesata.
    pesi: dizionario colonna → peso. Segno negativo = 'meno è meglio'."""
    if pesi is None:
        pesi = {"sharpe": 3, "ter_%": -2, "max_dd_%": 1, "vol_%": -1}

    out = tab.copy()
    contributi = pd.DataFrame(index=tab.index)

    for col, w in pesi.items():
        v = tab[col].astype(float)
        span = v.max() - v.min()
        norm = 50.0 if span == 0 else (v - v.min()) / span * 100   # 0-100
        contributi[col] = norm * w

    out["punteggio"] = contributi.sum(axis=1).round(1)
    out.attrs = tab.attrs
    out.attrs["pesi"] = pesi
    return out.sort_values("punteggio", ascending=False), contributi.round(1)

p, contrib = punteggio(res)
print("PESI:", p.attrs["pesi"])
print("PERIODO:", p.attrs["periodo"], "|", p.attrs["giorni"], "giorni |", p.attrs["fonte"])
p[["punteggio", "sharpe", "ter_%", "max_dd_%", "vol_%", "tipo"]]

In [ ]:
scenari = {
    "solo costi":       {"ter_%": -1},
    "solo Sharpe":      {"sharpe": 1},
    "prudente":         {"sharpe": 1, "max_dd_%": 3, "vol_%": -2, "ter_%": -1},
    "cacciatore":       {"sharpe": 4, "ter_%": -1},
    "default":          {"sharpe": 3, "ter_%": -2, "max_dd_%": 1, "vol_%": -1},
}

ordini = {}
for nome, w in scenari.items():
    pp, _ = punteggio(res, w)
    ordini[nome] = list(pp.index)

pd.DataFrame(ordini, index=[f"{i+1}°" for i in range(len(res))])

In [ ]:
print("Contributo di ogni colonna al punteggio (pesi default):\n")
print(contrib.loc[p.index])

In [ ]:
def punteggio(tab, pesi=None, robusto=True):
    """robusto=True: usa il rango invece del min-max.
    Immune agli outlier e alle scale schiacciate. Il costo: perde
    l'informazione su QUANTO uno è distante, tiene solo l'ordine."""
    if pesi is None:
        pesi = {"sharpe": 3, "ter_%": -2, "max_dd_%": 1, "vol_%": -1}

    out, contrib = tab.copy(), pd.DataFrame(index=tab.index)
    n = len(tab)
    for col, w in pesi.items():
        v = tab[col].astype(float)
        if robusto:
            norm = v.rank(method="average") / n * 100   # 20..100 con n=5
        else:
            span = v.max() - v.min()
            norm = 50.0 if span == 0 else (v - v.min()) / span * 100
        contrib[col] = norm * w

    tot = sum(abs(w) for w in pesi.values())
    out["punteggio"] = (contrib.sum(axis=1) / tot).round(1)   # scala confrontabile
    out.attrs = {**tab.attrs, "pesi": pesi, "metodo": "rango" if robusto else "min-max"}
    return out.sort_values("punteggio", ascending=False), contrib.round(1)

p, contrib = punteggio(res)
print(p.attrs["metodo"], "|", p.attrs["pesi"])
p[["punteggio","sharpe","ter_%","max_dd_%","vol_%","tipo"]]

In [ ]:
# quali differenze di Sharpe sono distinguibili dal rumore? (dalla cella 9)
p["sharpe_vs_bench"] = ["—" if t=="XDWD.DE" else "non distinguibile" for t in p.index]
p[["punteggio","sharpe","sharpe_vs_bench","ter_%","tipo"]]

In [ ]:
import feedparser, pandas as pd

FEED = {
    "ESG Today":   "https://www.esgtoday.com/feed/",
    "ESG Dive":    "https://www.esgdive.com/feeds/news/",
    "ESG News":    "https://esgnews.com/feed/",
}

def news(feeds=FEED, per_feed=8):
    righe = []
    for nome, url in feeds.items():
        try:
            f = feedparser.parse(url)
            if not f.entries:
                righe.append({"fonte": nome, "titolo": "[nessuna voce — feed cambiato?]",
                              "data": None, "link": url}); continue
            for e in f.entries[:per_feed]:
                righe.append({"fonte": nome, "titolo": e.get("title","")[:90],
                              "data": e.get("published", "")[:16], "link": e.get("link","")})
        except Exception as ex:
            righe.append({"fonte": nome, "titolo": f"[errore: {str(ex)[:40]}]",
                          "data": None, "link": url})
    return pd.DataFrame(righe)

n = news()
print(f"{len(n)} voci | scaricate il {pd.Timestamp.now():%Y-%m-%d %H:%M}")
n

In [ ]:
%pip install feedparser

In [6]:
import feedparser

FEED_ESG = {
    "ESG Today":       "https://www.esgtoday.com/feed/",
    "ESG News":        "https://esgnews.com/feed/",
    "ESG Investor":    "https://esginvestor.net/feed/",
    "KnowESG":         "https://www.knowesg.com/rss",
    "Practical ESG":   "https://practicalesg.com/blogs/feed/",
    "ESG Dive":        "https://www.esgdive.com/feeds/news.rss",
    "Trellis":         "https://trellis.net/feed/",
}

FEED_MERCATI = {
    "CNBC Markets":     "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=20910258",
    "CNBC Finance":     "https://search.cnbc.com/rs/search/combinedcms/view.xml?partnerId=wrss01&id=10000664",
    "MarketWatch":      "https://feeds.content.dowjones.io/public/rss/mw_topstories",
    "FT Markets":       "https://www.ft.com/markets?format=rss",
    "Economist Fin.":   "https://www.economist.com/finance-and-economics/rss.xml",
    "Klement Investing":"https://klementoninvesting.substack.com/feed",
    "Abnormal Returns": "https://abnormalreturns.com/feed/",
    "FRED Blog":        "https://fredblog.stlouisfed.org/feed/",
}

FEED = {**FEED_ESG, **FEED_MERCATI}

def news(feeds=FEED, per_feed=5):
    righe = []
    for nome, url in feeds.items():
        try:
            f = feedparser.parse(url)
            if not f.entries:
                righe.append({"fonte": nome, "titolo": "[nessuna voce]", "data": None, "link": url})
                continue
            for e in f.entries[:per_feed]:
                righe.append({"fonte": nome, "titolo": e.get("title","")[:85],
                              "data": e.get("published","")[:16], "link": e.get("link","")})
        except Exception as ex:
            righe.append({"fonte": nome, "titolo": f"[errore: {str(ex)[:35]}]", "data": None, "link": url})
    return pd.DataFrame(righe)

n = news()
ok = n[~n["titolo"].str.startswith("[")]
ko = n[n["titolo"].str.startswith("[")]

print(f"VIVI: {ok['fonte'].nunique()}/{len(FEED)} feed — {len(ok)} notizie")
print(f"MORTI: {list(ko['fonte'])}\n")
ok

VIVI: 12/15 feed — 60 notizie
MORTI: ['ESG Investor', 'KnowESG', 'ESG Dive']



,fonte,titolo,data,link
0,ESG Today,World’s Largest Beef Producer JBS Drops Supply...,"Fri, 10 Jul 2026",https://www.esgtoday.com/worlds-largest-beef-p...
1,ESG Today,Fleek Raises $25 Million to Scale Second-Hand ...,"Thu, 09 Jul 2026",https://www.esgtoday.com/fleek-raises-25-milli...
2,ESG Today,Canada Proposes New Oil and Gas Production Dec...,"Thu, 09 Jul 2026",https://www.esgtoday.com/canada-proposes-new-o...
3,ESG Today,"EthiFinance, ESG Book Merge to Form European S...","Thu, 09 Jul 2026",https://www.esgtoday.com/ethifinance-esg-book-...
4,ESG Today,Airbus Partners with MTU to Develop Hydrogen-P...,"Thu, 09 Jul 2026",https://www.esgtoday.com/airbus-partners-with-...
5,ESG News,Quinbrook Secures $780 Million For UK Renewabl...,"Fri, 10 Jul 2026",https://esgnews.com/quinbrook-secures-780-mill...
6,ESG News,Airbus and MTU To Develop First Fully Electric...,"Fri, 10 Jul 2026",https://esgnews.com/airbus-mtu-aero-engines-de...
7,ESG News,Fleek Secures $25 Million For Secondhand Fashi...,"Fri, 10 Jul 2026",https://esgnews.com/fleek-secures-25-million-f...
8,ESG News,Yesterday’s ESG Market Signals Today,"Thu, 09 Jul 2026",https://esgnews.com/yesterdays-esg-market-sign...
9,ESG News,How Beverage Companies Are Using Technology to...,"Thu, 09 Jul 2026",https://esgnews.com/how-beverage-companies-are...


In [7]:
codice = '''
"""Progetto Super — motore di misura.
Misura ed espone. Non prevede. Ogni numero porta l'etichetta di dove viene.
"""
import yfinance as yf
import pandas as pd
import numpy as np

GIORNI_BORSA = 252


def valida(tickers, inizio="2019-01-01", soglia_zeri=0.02):
    """Quali ticker esistono, hanno storico, e non hanno prezzi fermi?"""
    righe, scartati = [], []
    for t in tickers:
        try:
            d = yf.download(t, start=inizio, auto_adjust=True, progress=False)
            if isinstance(d.columns, pd.MultiIndex):
                d.columns = d.columns.droplevel("Ticker")
            s = d["Close"].dropna()
            if len(s) < 250:
                scartati.append((t, f"solo {len(s)} giorni")); continue
            zeri = int((s.pct_change().dropna() == 0).sum())
            righe.append({"ticker": t, "giorni": len(s), "zeri": zeri,
                          "%zeri": round(100 * zeri / len(s), 1),
                          "da": s.index.min().date(),
                          "usabile": zeri / len(s) < soglia_zeri})
        except Exception as e:
            scartati.append((t, str(e)[:40]))
    return pd.DataFrame(righe), scartati


def confronta(tickers, inizio="2019-01-01", fine=None, rf_annuo=0.0):
    """Scarica N ticker, li ALLINEA sui giorni comuni, ne calcola le metriche."""
    grezzi = yf.download(tickers, start=inizio, end=fine, auto_adjust=True, progress=False)
    px = grezzi["Close"].dropna()
    rend = px.pct_change().dropna()
    anni = len(rend) / GIORNI_BORSA
    rf_g = (1 + rf_annuo) ** (1 / GIORNI_BORSA) - 1
    righe = []
    for t in px.columns:
        r = rend[t]
        curva = (1 + r).cumprod()
        dd = curva / curva.cummax() - 1
        ex = r - rf_g
        righe.append({
            "ticker": t,
            "cagr_%": round(((px[t].iloc[-1] / px[t].iloc[0]) ** (1 / anni) - 1) * 100, 2),
            "vol_%": round(r.std() * np.sqrt(GIORNI_BORSA) * 100, 2),
            "sharpe": round(ex.mean() / ex.std() * np.sqrt(GIORNI_BORSA), 2),
            "max_dd_%": round(dd.min() * 100, 2),
            "gg_sott_acqua": int((dd < -0.0001).sum()),
            "zeri": int((r == 0).sum()),
        })
    tab = pd.DataFrame(righe).set_index("ticker")
    tab.attrs["periodo"] = f"{rend.index.min().date()} -> {rend.index.max().date()}"
    tab.attrs["giorni"] = len(rend)
    tab.attrs["rf_%"] = rf_annuo * 100
    tab.attrs["fonte"] = "yahoo / Close auto_adjust"
    return tab, px


def punteggio(tab, pesi=None, robusto=True):
    """Ordina secondo i pesi scelti. robusto=True usa il rango (immune agli outlier).
    NON dice quanto uno e' meglio. Dice solo in che ordine stanno, dati quei pesi."""
    if pesi is None:
        pesi = {"sharpe": 3, "ter_%": -2, "max_dd_%": 1, "vol_%": -1}
    out, contrib = tab.copy(), pd.DataFrame(index=tab.index)
    n = len(tab)
    for col, w in pesi.items():
        v = tab[col].astype(float)
        if robusto:
            norm = v.rank(method="average") / n * 100
        else:
            span = v.max() - v.min()
            norm = 50.0 if span == 0 else (v - v.min()) / span * 100
        contrib[col] = norm * w
    tot = sum(abs(w) for w in pesi.values())
    out["punteggio"] = (contrib.sum(axis=1) / tot).round(1)
    out.attrs = {**tab.attrs, "pesi": pesi, "metodo": "rango" if robusto else "min-max"}
    return out.sort_values("punteggio", ascending=False), contrib.round(1)


def delta_sharpe(prezzi, bench, n_boot=2000, seme=42):
    """Ogni ticker vs benchmark: la differenza di Sharpe e' distinguibile da zero?
    Bootstrap APPAIATO: ricampiona righe intere, cosi' la correlazione resta intatta.
    Limite: assume giorni indipendenti. L'intervallo vero e' piu' largo."""
    rng = np.random.default_rng(seme)
    r = prezzi.pct_change().dropna()
    n = len(r)
    sh = lambda s: s.mean() / s.std() * np.sqrt(GIORNI_BORSA)
    righe = []
    for t in r.columns:
        if t == bench:
            continue
        oss = sh(r[t]) - sh(r[bench])
        d = np.empty(n_boot)
        for i in range(n_boot):
            c = r.iloc[rng.integers(0, n, n)]
            d[i] = sh(c[t]) - sh(c[bench])
        lo, hi = np.percentile(d, [5, 95])
        righe.append({"ticker": t, "d_sharpe": round(oss, 3),
                      "ic90_lo": round(lo, 3), "ic90_hi": round(hi, 3),
                      "distinguibile": not (lo < 0 < hi)})
    tab = pd.DataFrame(righe).set_index("ticker")
    tab.attrs = {"bench": bench, "n_boot": n_boot, "giorni": n,
                 "metodo": "bootstrap appaiato su giorni singoli"}
    return tab
'''

with open("super.py", "w") as f:
    f.write(codice)
print("scritto super.py —", len(codice.splitlines()), "righe")

scritto super.py — 108 righe


In [8]:
from importlib import reload
import super as sp
reload(sp)   # ricarica il file se lo modifichi, senza riavviare il kernel

universo = {
    "XDWD.DE": ("MSCI World",          "benchmark",  0.12),
    "EUNL.DE": ("MSCI World",          "benchmark",  0.20),
    "XZW0.DE": ("MSCI World ESG",      "esg",        0.20),
    "2B7K.DE": ("MSCI World SRI",      "esg",        0.20),
    "IQQW.DE": ("MSCI World Screened", "esg",        0.20),
}

v, ko = sp.valida(list(universo))
buoni = v.loc[v["usabile"], "ticker"].tolist()
res, px_u = sp.confronta(buoni, inizio="2019-02-28")
res["ter_%"] = [universo[t][2] for t in res.index]
res["tipo"]  = [universo[t][1] for t in res.index]

p, contrib = sp.punteggio(res)
ds = sp.delta_sharpe(px_u, bench="XDWD.DE")

finale = p.join(ds[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])
print(finale.attrs["periodo"], "|", finale.attrs["giorni"], "gg |", finale.attrs["pesi"])
finale[["punteggio","sharpe","d_sharpe","ic90_lo","ic90_hi","distinguibile","ter_%","tipo"]]

KeyError: 'periodo'

In [9]:
from pathlib import Path
import datetime as dt

def archivia(df_news, cartella="news"):
    """Salva le notizie con la data in cui LE ABBIAMO VISTE.
    Serve per l'event study: se non sai quando l'informazione era pubblica,
    non puoi calcolare un rendimento anomalo."""
    Path(cartella).mkdir(exist_ok=True)
    df = df_news.copy()
    df["visto_il"] = dt.datetime.now().isoformat(timespec="seconds")
    f = Path(cartella) / "archivio.csv"
    if f.exists():
        vecchio = pd.read_csv(f)
        df = pd.concat([vecchio, df]).drop_duplicates(subset=["link"], keep="first")
    df.to_csv(f, index=False)
    return len(df), f

tot, dove = archivia(ok)
print(f"archivio: {tot} notizie uniche in {dove}")

archivio: 60 notizie uniche in news/archivio.csv


In [10]:
p, contrib = sp.punteggio(res)
ds = sp.delta_sharpe(px_u, bench="XDWD.DE")

finale = p.join(ds[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])
finale.attrs = {**p.attrs, "bench": ds.attrs["bench"], "boot": ds.attrs["metodo"]}

print(finale.attrs["periodo"], "|", finale.attrs["giorni"], "gg |", finale.attrs["pesi"])
print("bench:", finale.attrs["bench"], "|", finale.attrs["boot"])
finale[["punteggio","sharpe","d_sharpe","ic90_lo","ic90_hi","distinguibile","ter_%","tipo"]]

2019-03-01 -> 2026-07-10 | 1868 gg | {'sharpe': 3, 'ter_%': -2, 'max_dd_%': 1, 'vol_%': -1}
bench: XDWD.DE | bootstrap appaiato su giorni singoli


,punteggio,sharpe,d_sharpe,ic90_lo,ic90_hi,distinguibile,ter_%,tipo
ticker,,,,,,,,
XDWD.DE,34.3,0.88,NaN,NaN,NaN,NaN,0.12,benchmark
EUNL.DE,17.1,0.88,-0.000,-0.046,0.038,False,0.20,benchmark
IQQW.DE,5.7,0.87,-0.015,-0.067,0.034,False,0.20,esg
XZW0.DE,-5.7,0.86,-0.023,-0.135,0.085,False,0.20,esg
2B7K.DE,-8.6,0.84,-0.047,-0.184,0.095,False,0.20,esg


In [11]:
def bandiere(tab, prezzi, bench, ter_bench=None):
    """Non toccano il punteggio. Dicono cosa monitorare.
    Una bandiera che non si accende e' informazione anche lei."""
    r = prezzi.pct_change().dropna()
    out = {}
    for t in tab.index:
        f = []
        te = (r[t] - r[bench]).std() * np.sqrt(252) * 100
        corr = r[t].corr(r[bench])

        if t != bench and te < 1.5:
            f.append(f"TE {te:.2f}% — indistinguibile dal benchmark: paghi il filtro, compri l'indice")
        if corr > 0.99 and t != bench:
            f.append(f"corr {corr:.3f} — nessun beneficio di diversificazione")
        if tab.loc[t,"vol_%"] > tab.loc[bench,"vol_%"]:
            f.append(f"vol {tab.loc[t,'vol_%']:.2f}% > bench {tab.loc[bench,'vol_%']:.2f}% — meno titoli, meno diversificazione")
        if ter_bench and tab.loc[t,"ter_%"] > ter_bench:
            extra = (tab.loc[t,"ter_%"] - ter_bench)
            f.append(f"+{extra:.2f}pp di TER sul benchmark = {extra*10:.1f}€/anno ogni 1000€ investiti")
        if tab.loc[t,"zeri"] > 0.01 * len(r):
            f.append(f"{tab.loc[t,'zeri']} giorni a prezzo fermo — metriche di rischio inaffidabili")

        out[t] = f if f else ["— nessuna bandiera"]
    return out

b = bandiere(finale, px_u, "XDWD.DE", ter_bench=0.12)
for t, fl in b.items():
    print(f"\n{t}  ({finale.loc[t,'tipo']})")
    for x in fl:
        print("   •", x)


XDWD.DE  (benchmark)
   • — nessuna bandiera

EUNL.DE  (benchmark)
   • TE 1.05% — indistinguibile dal benchmark: paghi il filtro, compri l'indice
   • corr 0.998 — nessun beneficio di diversificazione
   • +0.08pp di TER sul benchmark = 0.8€/anno ogni 1000€ investiti

IQQW.DE  (esg)
   • TE 1.25% — indistinguibile dal benchmark: paghi il filtro, compri l'indice
   • corr 0.997 — nessun beneficio di diversificazione
   • +0.08pp di TER sul benchmark = 0.8€/anno ogni 1000€ investiti

XZW0.DE  (esg)
   • vol 16.70% > bench 16.13% — meno titoli, meno diversificazione
   • +0.08pp di TER sul benchmark = 0.8€/anno ogni 1000€ investiti
   • 21 giorni a prezzo fermo — metriche di rischio inaffidabili

2B7K.DE  (esg)
   • vol 16.16% > bench 16.13% — meno titoli, meno diversificazione
   • +0.08pp di TER sul benchmark = 0.8€/anno ogni 1000€ investiti
   • 21 giorni a prezzo fermo — metriche di rischio inaffidabili


In [12]:
CAPITALE = 10_000
anni = finale.attrs["giorni"] / 252
bench = "XDWD.DE"

print(f"{CAPITALE:,}€ investiti per {anni:.1f} anni ({finale.attrs['periodo']})\n")
print(f"{'ticker':10s} {'valore finale':>14s} {'vs bench':>10s} {'tipo':>10s}")
for t in finale.index:
    fin = CAPITALE * (1 + finale.loc[t,"cagr_%"]/100) ** anni
    ref = CAPITALE * (1 + finale.loc[bench,"cagr_%"]/100) ** anni
    print(f"{t:10s} {fin:>13,.0f}€ {fin-ref:>+9,.0f}€ {finale.loc[t,'tipo']:>10s}")

10,000€ investiti per 7.4 anni (2019-03-01 -> 2026-07-10)

ticker      valore finale   vs bench       tipo
XDWD.DE           26,088€        +0€  benchmark
EUNL.DE           26,071€       -17€  benchmark
IQQW.DE           25,516€      -572€        esg
XZW0.DE           26,139€       +51€        esg
2B7K.DE           24,694€    -1,394€        esg


In [13]:
def universo_usa():
    """Listing ufficiale NASDAQ Trader, aggiornato ogni notte. Fonte primaria."""
    url = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqtraded.txt"
    df = pd.read_csv(url, sep="|")
    df = df[df["Symbol"].notna() & ~df["Symbol"].str.contains("File Creation", na=False)]
    out = pd.DataFrame({"ticker": df["Symbol"], "nome": df["Security Name"],
                        "tipo": df["ETF"].map({"Y":"etf","N":"azione"}), "test": df["Test Issue"]})
    out = out[out["test"]=="N"].drop(columns="test")
    out.attrs["url"] = url
    out.attrs["scaricato"] = pd.Timestamp.now().isoformat(timespec="minutes")
    return out

u = universo_usa()
print(u.attrs["scaricato"], "|", len(u), "strumenti")
print(u["tipo"].value_counts().to_dict())
u.sample(6)

2026-07-10T14:24 | 12990 strumenti
{'azione': 7476, 'etf': 5514}


,ticker,nome,tipo
7029,LRCC,Corgi LRCX 2x Daily ETF,etf
6096,IRS,IRSA Inversiones Y Representaciones S.A. Globa...,azione
60,ABEV,Ambev S.A. American Depositary Shares (Each re...,azione
10528,SLVR,Sprott Silver Miners & Physical Silver ETF,etf
7887,NCNO,"nCino, Inc. - Common Stock",azione
4372,FMCE,FM Compounders Equity ETF,etf


In [14]:
def cerca(df, *parole, tipo=None):
    """Filtra per parole nel nome. Il nome e' un dato pubblico, non un rating."""
    m = pd.Series(True, index=df.index)
    for p in parole:
        m &= df["nome"].str.contains(p, case=False, na=False)
    if tipo:
        m &= df["tipo"] == tipo
    return df[m]

esg = cerca(u, "ESG", tipo="etf")
print(f"{len(esg)} ETF con 'ESG' nel nome\n")
print(cerca(u, "S&P 500", tipo="etf")[["ticker","nome"]].head(10).to_string(index=False))

51 ETF con 'ESG' nel nome

ticker                                                   nome
  APRP                     PGIM S&P 500 Buffer 12 ETF - April
  AUGP                    PGIM S&P 500 Buffer 12 ETF - August
   BBB    CYBER HORNET S&P 500 and Bitcoin 75/25 Strategy ETF
  BUFP                    PGIM Laddered S&P 500 Buffer 12 ETF
  BUYB              ProShares S&P 500 Buyback Aristocrats ETF
  CATH                   Global X S&P 500 Catholic Values ETF
  CHRI                  Global X S&P 500 Christian Values ETF
  CPSA  Calamos S&P 500 Structured Alt Protection ETF  August
  CPSD Calamos S&P 500 Structured Alt Protection ETF December
  CPSF Calamos S&P 500 Structured Alt Protection ETF February


In [15]:
# Cerchiamo ETF S&P 500 puri: escludiamo i mascherati
esg = cerca(u, "ESG", tipo="etf")
sp500 = cerca(u, "S&P 500", tipo="etf")

VELENI = ["Buffer","Bitcoin","2x","3x","Inverse","Ultra","Covered Call",
          "Buyback","Values","Structured","Laddered","Hedged","Protection"]

def pulisci(df, veleni=VELENI):
    """Toglie leva, opzioni, tematici, strutturati. Non e' un giudizio:
    e' che uno strumento con leva 2x non e' confrontabile con uno senza."""
    m = pd.Series(True, index=df.index)
    for v in veleni:
        m &= ~df["nome"].str.contains(v, case=False, na=False)
    return df[m]

sp500_puri = pulisci(sp500)
esg_puri   = pulisci(esg)

print(f"S&P 500:  {len(sp500)} → {len(sp500_puri)} dopo il filtro")
print(f"ESG:      {len(esg)} → {len(esg_puri)} dopo il filtro\n")
print(sp500_puri[["ticker","nome"]].to_string(index=False))

S&P 500:  160 → 87 dopo il filtro
ESG:      51 → 51 dopo il filtro

ticker                                                     nome
  DIVG                Invesco S&P 500 High Dividend Growers ETF
  DRVR                     Amplify S&P 500 Dividend Drivers ETF
  DSPY              Tema S&P 500 Historical Weight ETF Strategy
   EEE     CYBER HORNET S&P 500 and Ethereum 75/25 Strategy ETF
  EFIV                        State Street SPDR S&P 500 ESG ETF
  EGLE                Global X S&P 500 U.S. Revenue Leaders ETF
  EMOT                    First Trust S&P 500 Economic Moat ETF
  FCFY       First Trust S&P 500 Diversified Free Cash Flow ETF
  FLAG          Global X S&P 500 U.S. Market Leaders Top 50 ETF
  GPIX                 Goldman Sachs S&P 500 Premium Income ETF
  ISPY                        ProShares S&P 500 High Income ETF
   IVE                                iShares S&P 500 Value ETF
   IVV                                 iShares Core S&P 500 ETF
  IVVW                             i

In [16]:
import yfinance as yf

def sopravvivono(tickers, inizio="2020-01-01", max_zeri=0.02, min_giorni=1000):
    """Il nome mente. I dati no. Chi ha storico e scambia davvero?"""
    ok, ko = [], []
    for t in tickers:
        try:
            d = yf.download(t, start=inizio, auto_adjust=True, progress=False)
            if isinstance(d.columns, pd.MultiIndex):
                d.columns = d.columns.droplevel("Ticker")
            s = d["Close"].dropna()
            if len(s) < min_giorni:
                ko.append((t, f"{len(s)} gg")); continue
            r = s.pct_change().dropna()
            z = (r == 0).mean()
            if z > max_zeri:
                ko.append((t, f"{z*100:.1f}% zeri")); continue
            ok.append({"ticker": t, "giorni": len(s), "%zeri": round(z*100,1),
                       "vol_medio_$": round(float((d["Close"]*d["Volume"]).median())/1e6, 1)})
        except Exception as e:
            ko.append((t, str(e)[:25]))
    return pd.DataFrame(ok).sort_values("vol_medio_$", ascending=False), ko

cand = sp500_puri["ticker"].tolist()[:20]
vivi, morti = sopravvivono(cand)
print(f"{len(vivi)} vivi / {len(cand)} testati")
print("scartati:", morti[:5], "...\n")
vivi

6 vivi / 20 testati
scartati: [('DIVG', '647 gg'), ('DRVR', '1 gg'), ('DSPY', '318 gg'), ('EEE', '110 gg'), ('EGLE', '308 gg')] ...



,ticker,giorni,%zeri,vol_medio_$
2,IVV,1637,0.0,2040.9
3,IVW,1637,0.5,154.0
1,IVE,1637,0.6,99.2
5,NOBL,1637,0.6,44.1
4,KNG,1637,0.5,3.8
0,EFIV,1493,0.5,1.2


In [17]:
FAMIGLIE = {
    "IVV":  ("benchmark",  "nessuna esclusione"),
    "SPY":  ("benchmark",  "nessuna esclusione"),
    "EFIV": ("esg",        "ESG score S&P"),
    "SPYX": ("esclusione", "riserve fossili"),
    "SPXE": ("esclusione", "settore energia"),
    "SPUS": ("religiosa",  "sharia"),
    "CATH": ("religiosa",  "cattolica"),
}

vivi2, morti2 = sopravvivono(list(FAMIGLIE), inizio="2020-01-01", min_giorni=1200)
print(f"vivi: {list(vivi2['ticker'])}\nscartati: {morti2}\n")
vivi2

vivi: ['SPY', 'IVV', 'SPYX', 'SPUS', 'EFIV', 'CATH', 'SPXE']
scartati: []



,ticker,giorni,%zeri,vol_medio_$
1,SPY,1637,0.1,31888.9
0,IVV,1637,0.0,2040.9
3,SPYX,1637,0.7,3.0
5,SPUS,1637,1.2,1.4
2,EFIV,1493,0.5,1.2
6,CATH,1637,0.6,1.2
4,SPXE,1637,0.2,0.0


In [18]:
buoni = vivi2["ticker"].tolist()
res2, px2 = sp.confronta(buoni, inizio="2020-06-01")
res2["famiglia"] = [FAMIGLIE[t][0] for t in res2.index]
res2["esclude"]  = [FAMIGLIE[t][1] for t in res2.index]

r2 = px2.pct_change().dropna()
bench = "IVV"
res2["te_%"]  = [round((r2[t]-r2[bench]).std()*np.sqrt(252)*100, 2) for t in res2.index]
res2["corr"]  = [round(r2[t].corr(r2[bench]), 4) for t in res2.index]

ds2 = sp.delta_sharpe(px2, bench=bench, n_boot=1000)
f2 = res2.join(ds2[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])

print(res2.attrs["periodo"], "|", res2.attrs["giorni"], "gg | bench:", bench)
f2[["cagr_%","vol_%","sharpe","d_sharpe","ic90_lo","ic90_hi","distinguibile","te_%","famiglia","esclude"]].sort_values("te_%")

2020-07-30 -> 2026-07-09 | 1492 gg | bench: IVV


,cagr_%,vol_%,sharpe,d_sharpe,ic90_lo,ic90_hi,distinguibile,te_%,famiglia,esclude
ticker,,,,,,,,,,
IVV,16.90,16.70,1.02,NaN,NaN,NaN,NaN,0.00,benchmark,nessuna esclusione
SPY,16.82,16.85,1.01,-0.011,-0.038,0.013,False,0.70,benchmark,nessuna esclusione
SPYX,16.50,16.85,0.99,-0.028,-0.093,0.037,False,1.60,esclusione,riserve fossili
EFIV,17.31,16.84,1.03,0.014,-0.085,0.110,False,2.37,esg,ESG score S&P
SPXE,16.52,16.71,1.00,-0.020,-0.115,0.075,False,2.41,esclusione,settore energia
CATH,15.70,17.61,0.92,-0.102,-0.306,0.076,False,4.86,religiosa,cattolica
SPUS,18.79,19.17,0.99,-0.025,-0.195,0.161,False,5.46,religiosa,sharia


In [19]:
# 1. il TE spiega il d_sharpe?
sub = f2.dropna(subset=["d_sharpe"])
c = sub["te_%"].corr(sub["d_sharpe"])
print(f"corr(TE, ΔSharpe) = {c:.3f}   su {len(sub)} ETF\n")

# 2. e il tilt fattoriale? regrediamo ogni spread su energia e finanziari
fatt = sp.confronta(["XLE","XLF","XLK","IVV"], inizio="2020-07-30")[1].pct_change().dropna()
r2 = px2.pct_change().dropna()
com = r2.index.intersection(fatt.index)

print(f"{'ticker':6s} {'energia':>9s} {'finanz':>9s} {'tech':>9s} {'R²':>7s}   esclude")
for t in f2.index:
    if t == "IVV": continue
    y = (r2.loc[com,t] - r2.loc[com,"IVV"]).values
    X = np.column_stack([np.ones(len(com)),
                         (fatt.loc[com,"XLE"] - fatt.loc[com,"IVV"]).values,
                         (fatt.loc[com,"XLF"] - fatt.loc[com,"IVV"]).values,
                         (fatt.loc[com,"XLK"] - fatt.loc[com,"IVV"]).values])
    b, *_ = np.linalg.lstsq(X, y, rcond=None)
    p = X @ b
    r2_ = 1 - ((y-p)**2).sum()/((y-y.mean())**2).sum()
    print(f"{t:6s} {b[1]:+9.4f} {b[2]:+9.4f} {b[3]:+9.4f} {r2_:7.3f}   {f2.loc[t,'esclude']}")

corr(TE, ΔSharpe) = -0.552   su 6 ETF

ticker   energia    finanz      tech      R²   esclude
CATH     -0.0004   +0.0373   +0.0338   0.006   cattolica
EFIV     -0.0011   -0.0127   +0.0222   0.028   ESG score S&P
SPUS     -0.0146   -0.1531   +0.2165   0.602   sharia
SPXE     -0.0369   +0.0083   -0.0039   0.143   settore energia
SPY      +0.0021   +0.0029   +0.0125   0.030   nessuna esclusione
SPYX     -0.0257   +0.0094   +0.0040   0.166   riserve fossili


In [20]:
def collega(news_df, universo, min_len=5):
    """Collega ogni notizia ai ticker citati. Matching di stringa sul nome
    ufficiale. Nessun giudizio, nessun LLM: solo 'questo nome compare nel titolo?'
    Falsificabile: apri la notizia e verifichi."""
    az = universo[universo["tipo"]=="azione"].copy()
    az["chiave"] = (az["nome"].str.split(" - ").str[0]
                              .str.replace(r"\b(Inc|Corp|Corporation|Ltd|plc|S\.A\.|N\.V\.|Company|Co)\b\.?",
                                           "", regex=True).str.strip())
    az = az[az["chiave"].str.len() >= min_len]

    righe = []
    for _, n in news_df.iterrows():
        for _, a in az.iterrows():
            if a["chiave"].lower() in n["titolo"].lower():
                righe.append({"ticker": a["ticker"], "azienda": a["chiave"],
                              "titolo": n["titolo"], "fonte": n["fonte"],
                              "data": n["data"], "link": n["link"]})
    out = pd.DataFrame(righe)
    out.attrs["metodo"] = "match esatto del nome ufficiale nel titolo"
    out.attrs["limite"] = "perde alias (Meta≠Facebook), sigle, e nomi comuni"
    return out

link = collega(ok, u)
print(f"{len(link)} collegamenti su {len(ok)} notizie")
print(link.attrs["limite"])
link[["ticker","azienda","titolo","data"]].head(15)

15 collegamenti su 60 notizie
perde alias (Meta≠Facebook), sigle, e nomi comuni


,ticker,azienda,titolo,data
0,MSFT,Microsoft,Microsoft adjusts climate agenda as emissions ...,"Thu, 09 Jul 2026"
1,MSTR,Strategy,McDonald’s reassigns its chief sustainability ...,"Thu, 09 Jul 2026"
2,STRC,Strategy,McDonald’s reassigns its chief sustainability ...,"Thu, 09 Jul 2026"
3,STRD,Strategy,McDonald’s reassigns its chief sustainability ...,"Thu, 09 Jul 2026"
4,STRF,Strategy,McDonald’s reassigns its chief sustainability ...,"Thu, 09 Jul 2026"
5,STRK,Strategy,McDonald’s reassigns its chief sustainability ...,"Thu, 09 Jul 2026"
6,MSTR,Strategy,What’s your space strategy? Why organizations ...,"Thu, 09 Jul 2026"
7,STRC,Strategy,What’s your space strategy? Why organizations ...,"Thu, 09 Jul 2026"
8,STRD,Strategy,What’s your space strategy? Why organizations ...,"Thu, 09 Jul 2026"
9,STRF,Strategy,What’s your space strategy? Why organizations ...,"Thu, 09 Jul 2026"


In [21]:
app = '''
import streamlit as st
import pandas as pd, numpy as np
import super as sp

st.set_page_config(page_title="Progetto Super", layout="wide")
st.title("Progetto Super")
st.caption("Non dice cosa comprare. Dice cosa stai comprando.")

UNIVERSI = {
    "S&P 500 — screen a confronto": {
        "IVV":  ("benchmark",  "nessuna esclusione", 0.03),
        "SPY":  ("benchmark",  "nessuna esclusione", 0.09),
        "EFIV": ("esg",        "ESG score S&P",      0.10),
        "SPYX": ("esclusione", "riserve fossili",    0.20),
        "SPXE": ("esclusione", "settore energia",    0.27),
        "SPUS": ("religiosa",  "sharia",             0.45),
        "CATH": ("religiosa",  "cattolica",          0.29),
    },
    "MSCI World — UCITS": {
        "XDWD.DE": ("benchmark", "nessuna", 0.12),
        "EUNL.DE": ("benchmark", "nessuna", 0.20),
        "XZW0.DE": ("esg",       "MSCI ESG", 0.20),
        "2B7K.DE": ("esg",       "MSCI SRI", 0.20),
        "IQQW.DE": ("esg",       "Screened", 0.20),
    },
}

with st.sidebar:
    st.header("Universo")
    nome_u = st.selectbox("Gruppo confrontabile", list(UNIVERSI))
    universo = UNIVERSI[nome_u]
    bench = st.selectbox("Benchmark", list(universo))
    inizio = st.date_input("Da", pd.Timestamp("2020-07-30")).isoformat()

    st.header("Pesi del punteggio")
    st.caption("Spostali: la classifica si riordina. Vederlo e' il risultato.")
    w_sharpe = st.slider("Sharpe", 0, 5, 3)
    w_ter    = st.slider("TER (penalizza)", 0, 5, 2)
    w_dd     = st.slider("Max drawdown", 0, 5, 1)
    w_vol    = st.slider("Volatilita (penalizza)", 0, 5, 1)

@st.cache_data(ttl=3600)
def carica(tickers, inizio):
    return sp.confronta(list(tickers), inizio=inizio)

@st.cache_data(ttl=3600)
def boot(_px, bench):
    return sp.delta_sharpe(_px, bench=bench, n_boot=1000)

res, px = carica(tuple(universo), inizio)
res["ter_%"]   = [universo[t][2] for t in res.index]
res["famiglia"]= [universo[t][0] for t in res.index]
res["esclude"] = [universo[t][1] for t in res.index]

r = px.pct_change().dropna()
res["te_%"] = [round((r[t]-r[bench]).std()*np.sqrt(252)*100, 2) for t in res.index]

pesi = {"sharpe": w_sharpe, "ter_%": -w_ter, "max_dd_%": w_dd, "vol_%": -w_vol}
pesi = {k: v for k, v in pesi.items() if v != 0}
if not pesi:
    st.warning("Tutti i pesi a zero: nessun ordinamento possibile.")
    st.stop()

p, contrib = sp.punteggio(res, pesi)
ds = boot(px, bench)
tab = p.join(ds[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])

st.info(f"**{tab.attrs['periodo']}** — {tab.attrs['giorni']} giorni di borsa · "
        f"risk-free assunto {tab.attrs['rf_%']:.2f}% · fonte {tab.attrs['fonte']} · "
        f"normalizzazione: {tab.attrs['metodo']}")

col1, col2 = st.columns([3,2])

with col1:
    st.subheader("Classifica")
    st.caption(f"Punteggio = {pesi}. Cambia i pesi e guarda l'ordine muoversi.")
    vista = tab[["punteggio","sharpe","cagr_%","vol_%","max_dd_%","ter_%","te_%","famiglia","esclude"]]
    st.dataframe(vista, use_container_width=True)

with col2:
    st.subheader("Queste differenze sono reali?")
    st.caption(f"DeltaSharpe vs {bench}, IC 90% bootstrap appaiato, 1000 ricampionamenti.")
    for t in tab.index:
        if t == bench:
            st.write(f"**{t}** — benchmark")
            continue
        lo, hi, d = tab.loc[t,"ic90_lo"], tab.loc[t,"ic90_hi"], tab.loc[t,"d_sharpe"]
        if lo < 0 < hi:
            st.write(f"**{t}** {d:+.3f} [{lo:+.3f}; {hi:+.3f}] — :orange[non distinguibile dal rumore]")
        else:
            st.write(f"**{t}** {d:+.3f} [{lo:+.3f}; {hi:+.3f}] — :green[distinguibile]")
    st.caption("Il bootstrap su giorni singoli sottostima l'incertezza (autocorrelazione). "
               "L'intervallo vero e' piu' largo.")

st.divider()
st.subheader("Cosa monitorare")
for t in tab.index:
    f = []
    te, ter = tab.loc[t,"te_%"], tab.loc[t,"ter_%"]
    if t != bench and te < 1.5:
        f.append(f"TE {te:.2f}% — paghi il filtro, compri l'indice")
    if tab.loc[t,"vol_%"] > tab.loc[bench,"vol_%"]:
        f.append(f"vol {tab.loc[t,'vol_%']:.2f}% > bench — meno titoli, meno diversificazione")
    if ter > tab.loc[bench,"ter_%"]:
        d = ter - tab.loc[bench,"ter_%"]
        f.append(f"+{d:.2f}pp di TER = {d*100:.0f} euro/anno ogni 10.000 investiti")
    if tab.loc[t,"zeri"] > 0.01*tab.attrs["giorni"]:
        f.append(f"{tab.loc[t,'zeri']} giorni a prezzo fermo — metriche di rischio inaffidabili")
    if f:
        st.write(f"**{t}**")
        for x in f: st.write(f"  - {x}")

st.divider()
st.caption("Nessun numero qui e' una raccomandazione. Le decisioni restano di Carlo.")
'''

with open("app.py", "w") as fh:
    fh.write(app)
print("scritto app.py")

scritto app.py


In [22]:
%pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 21.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.6/797.6 kB 15.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 25.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 25.9 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18/18 [streamlit]18 [streamlit]
Note: you may need to restart the kernel to use updated packages.


In [1]:
with open("app.py") as f:
    vecchio = f.read()

nuovo = vecchio.replace(
    'tab = p.join(ds[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])',
    'tab = p.join(ds[["d_sharpe","ic90_lo","ic90_hi","distinguibile"]])\ntab.attrs = dict(p.attrs)'
)

if nuovo == vecchio:
    print("ATTENZIONE: nessuna sostituzione fatta — la riga non è stata trovata")
else:
    with open("app.py", "w") as f:
        f.write(nuovo)
    print("corretto. Cerco conferma:")
    print([l for l in nuovo.splitlines() if "tab.attrs = dict" in l])

corretto. Cerco conferma:
['tab.attrs = dict(p.attrs)', 'tab.attrs = dict(p.attrs)']


In [2]:
with open("app.py") as f:
    a = f.read()

# 1. bootstrap più leggero e cache stabile
a = a.replace(
    '@st.cache_data(ttl=3600)\ndef boot(_px, bench):\n    return sp.delta_sharpe(_px, bench=bench, n_boot=1000)',
    '@st.cache_data(ttl=3600)\ndef boot(tickers, inizio, bench):\n    _, px = sp.confronta(list(tickers), inizio=inizio)\n    return sp.delta_sharpe(px, bench=bench, n_boot=500)'
)
a = a.replace('ds = boot(px, bench)', 'ds = boot(tuple(universo), inizio, bench)')

with open("app.py", "w") as f:
    f.write(a)

print("boot ora è in cache su (tickers, inizio, bench)")
print([l for l in a.splitlines() if "def boot" in l or "ds = boot" in l])

boot ora è in cache su (tickers, inizio, bench)
['def boot(tickers, inizio, bench):', 'ds = boot(tuple(universo), inizio, bench)']


In [3]:
import time
from pathlib import Path

AREE = {
    "USA — S&P 500":   {"bench": "IVV", "cerca": ["S&P 500"]},
    "USA — total mkt": {"bench": "VTI", "cerca": ["Total Stock Market", "Total Market"]},
    "Europa":          {"bench": "VGK", "cerca": ["Europe", "Eurozone", "European"]},
    "Cina":            {"bench": "MCHI","cerca": ["China", "Chinese"]},
}

VELENI = ["Buffer","Bitcoin","Ethereum","Solana","XRP","2x","3x","4X","Inverse","Ultra",
          "Bear","Leveraged","Covered Call","BuyWrite","Structured","Laddered",
          "Target Income","Managed Distribution","ETN","Collar","Tail Risk"]

FAMIGLIA = {
    "esg":        ["ESG"],
    "sri":        ["SRI","Socially Responsible"],
    "screened":   ["Screened","Scored"],
    "clima":      ["Climate","Paris","Carbon","Fossil Fuel","Low Carbon","Clean Energy"],
    "religiosa":  ["Catholic","Christian","Sharia","Halal","Biblical"],
    "esclusione": ["Ex-","Free","Exclusions"],
}

def famiglia(nome):
    for f, parole in FAMIGLIA.items():
        if any(p.lower() in nome.lower() for p in parole):
            return f
    return "benchmark"

def costruisci(u, aree=AREE, veleni=VELENI, min_vol_M=5, min_giorni=750, max_per_area=60):
    """Testa i candidati su Yahoo UNA VOLTA. Lento. Poi salvato su CSV."""
    righe = []
    for area, cfg in aree.items():
        m = pd.Series(False, index=u.index)
        for p in cfg["cerca"]:
            m |= u["nome"].str.contains(p, case=False, na=False)
        cand = u[m & (u["tipo"]=="etf")].copy()
        for v in veleni:
            cand = cand[~cand["nome"].str.contains(v, case=False, na=False)]
        tick = list(dict.fromkeys([cfg["bench"]] + cand["ticker"].tolist()))[:max_per_area]
        print(f"{area}: testo {len(tick)} candidati...", end=" ", flush=True)

        vivi = 0
        for t in tick:
            try:
                d = yf.download(t, start="2019-01-01", auto_adjust=True, progress=False)
                if isinstance(d.columns, pd.MultiIndex):
                    d.columns = d.columns.droplevel("Ticker")
                s = d["Close"].dropna()
                if len(s) < min_giorni: continue
                r = s.pct_change().dropna()
                z = (r == 0).mean()
                if z > 0.02: continue
                vol = float((d["Close"]*d["Volume"]).median())/1e6
                if vol < min_vol_M: continue
                nome = u.loc[u["ticker"]==t, "nome"].iloc[0] if (u["ticker"]==t).any() else t
                righe.append({"ticker": t, "nome": nome[:60], "area": area,
                              "bench": cfg["bench"], "famiglia": famiglia(nome),
                              "giorni": len(s), "zeri_%": round(z*100,1),
                              "vol_M$": round(vol,1), "da": str(s.index.min().date())})
                vivi += 1
            except Exception:
                pass
        print(f"{vivi} sopravvissuti")
        time.sleep(1)

    df = pd.DataFrame(righe)
    df.to_csv("universo.csv", index=False)
    print(f"\nSALVATO universo.csv — {len(df)} strumenti")
    print(df.groupby(["area","famiglia"]).size())
    return df

uni = costruisci(u)
uni

NameError: name 'u' is not defined

In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
import time
from pathlib import Path

def universo_usa():
    url = "https://www.nasdaqtrader.com/dynamic/symdir/nasdaqtraded.txt"
    df = pd.read_csv(url, sep="|")
    df = df[df["Symbol"].notna() & ~df["Symbol"].str.contains("File Creation", na=False)]
    out = pd.DataFrame({"ticker": df["Symbol"], "nome": df["Security Name"],
                        "tipo": df["ETF"].map({"Y":"etf","N":"azione"}), "test": df["Test Issue"]})
    out = out[out["test"]=="N"].drop(columns="test")
    return out

u = universo_usa()
print(len(u), "strumenti |", u["tipo"].value_counts().to_dict())

12990 strumenti | {'azione': 7476, 'etf': 5514}


In [5]:
import time
from pathlib import Path

AREE = {
    "USA — S&P 500":   {"bench": "IVV", "cerca": ["S&P 500"]},
    "USA — total mkt": {"bench": "VTI", "cerca": ["Total Stock Market", "Total Market"]},
    "Europa":          {"bench": "VGK", "cerca": ["Europe", "Eurozone", "European"]},
    "Cina":            {"bench": "MCHI","cerca": ["China", "Chinese"]},
}

VELENI = ["Buffer","Bitcoin","Ethereum","Solana","XRP","2x","3x","4X","Inverse","Ultra",
          "Bear","Leveraged","Covered Call","BuyWrite","Structured","Laddered",
          "Target Income","Managed Distribution","ETN","Collar","Tail Risk"]

FAMIGLIA = {
    "esg":        ["ESG"],
    "sri":        ["SRI","Socially Responsible"],
    "screened":   ["Screened","Scored"],
    "clima":      ["Climate","Paris","Carbon","Fossil Fuel","Low Carbon","Clean Energy"],
    "religiosa":  ["Catholic","Christian","Sharia","Halal","Biblical"],
    "esclusione": ["Ex-","Free","Exclusions"],
}

def famiglia(nome):
    for f, parole in FAMIGLIA.items():
        if any(p.lower() in nome.lower() for p in parole):
            return f
    return "benchmark"

def costruisci(u, aree=AREE, veleni=VELENI, min_vol_M=5, min_giorni=750, max_per_area=60):
    """Testa i candidati su Yahoo UNA VOLTA. Lento. Poi salvato su CSV."""
    righe = []
    for area, cfg in aree.items():
        m = pd.Series(False, index=u.index)
        for p in cfg["cerca"]:
            m |= u["nome"].str.contains(p, case=False, na=False)
        cand = u[m & (u["tipo"]=="etf")].copy()
        for v in veleni:
            cand = cand[~cand["nome"].str.contains(v, case=False, na=False)]
        tick = list(dict.fromkeys([cfg["bench"]] + cand["ticker"].tolist()))[:max_per_area]
        print(f"{area}: testo {len(tick)} candidati...", end=" ", flush=True)

        vivi = 0
        for t in tick:
            try:
                d = yf.download(t, start="2019-01-01", auto_adjust=True, progress=False)
                if isinstance(d.columns, pd.MultiIndex):
                    d.columns = d.columns.droplevel("Ticker")
                s = d["Close"].dropna()
                if len(s) < min_giorni: continue
                r = s.pct_change().dropna()
                z = (r == 0).mean()
                if z > 0.02: continue
                vol = float((d["Close"]*d["Volume"]).median())/1e6
                if vol < min_vol_M: continue
                nome = u.loc[u["ticker"]==t, "nome"].iloc[0] if (u["ticker"]==t).any() else t
                righe.append({"ticker": t, "nome": nome[:60], "area": area,
                              "bench": cfg["bench"], "famiglia": famiglia(nome),
                              "giorni": len(s), "zeri_%": round(z*100,1),
                              "vol_M$": round(vol,1), "da": str(s.index.min().date())})
                vivi += 1
            except Exception:
                pass
        print(f"{vivi} sopravvissuti")
        time.sleep(1)

    df = pd.DataFrame(righe)
    df.to_csv("universo.csv", index=False)
    print(f"\nSALVATO universo.csv — {len(df)} strumenti")
    print(df.groupby(["area","famiglia"]).size())
    return df

uni = costruisci(u)
uni

USA — S&P 500: testo 60 candidati... 16 sopravvissuti
USA — total mkt: testo 2 candidati... 1 sopravvissuti
Europa: testo 29 candidati... 7 sopravvissuti
Cina: testo 47 candidati... 6 sopravvissuti

SALVATO universo.csv — 30 strumenti
area             famiglia 
Cina             benchmark     6
Europa           benchmark     7
USA — S&P 500    benchmark    15
                 screened      1
USA — total mkt  benchmark     1
dtype: int64


,ticker,nome,area,bench,famiglia,giorni,zeri_%,vol_M$,da
0,IVV,iShares Core S&P 500 ETF,USA — S&P 500,IVV,benchmark,1890,0.0,1847.6,2019-01-02
1,IVE,iShares S&P 500 Value ETF,USA — S&P 500,IVV,benchmark,1890,0.5,94.2,2019-01-02
2,IVW,iShares S&P 500 Growth ETF,USA — S&P 500,IVV,benchmark,1890,0.6,140.3,2019-01-02
3,NOBL,ProShares S&P 500 Dividend Aristocrats ETF,USA — S&P 500,IVV,benchmark,1890,0.6,41.5,2019-01-02
4,RPG,Invesco S&P 500 Pure Growth ETF,USA — S&P 500,IVV,benchmark,1890,0.6,9.4,2019-01-02
5,RPV,Invesco S&P 500 Pure Value ETF,USA — S&P 500,IVV,benchmark,1890,0.4,15.2,2019-01-02
6,RSP,Invesco S&P 500 Equal Weight ETF,USA — S&P 500,IVV,benchmark,1890,0.3,464.5,2019-01-02
7,RSPT,Invesco S&P 500 Equal Weight Technology ETF,USA — S&P 500,IVV,benchmark,1890,0.4,10.2,2019-01-02
8,RWL,Invesco S&P 500 Revenue ETF,USA — S&P 500,IVV,benchmark,1890,0.5,5.3,2019-01-02
9,SNPE,Xtrackers S&P 500 Scored & Screened ETF,USA — S&P 500,IVV,screened,1769,1.1,5.5,2019-06-26


In [6]:
# yfinance a volte ha expense ratio nel .info — spesso no. Proviamo, e dichiariamo i buchi.
def prova_ter(tickers):
    out = {}
    for t in tickers:
        try:
            info = yf.Ticker(t).info
            e = info.get("netExpenseRatio") or info.get("annualReportExpenseRatio")
            out[t] = round(float(e)*100, 3) if e and e < 1 else (round(float(e),3) if e else None)
        except Exception:
            out[t] = None
    return out

ter = prova_ter(uni["ticker"].tolist())
uni["ter_%"] = uni["ticker"].map(ter)

mancanti = uni["ter_%"].isna().sum()
print(f"TER trovato per {len(uni)-mancanti}/{len(uni)} — mancanti: {mancanti}")
print("\n7.13 del CFA: copertura sotto il 75% compromette l'analisi.")
print(f"copertura: {100*(len(uni)-mancanti)/len(uni):.0f}%")
uni.to_csv("universo.csv", index=False)
uni[uni["ter_%"].isna()][["ticker","nome"]]

TER trovato per 30/30 — mancanti: 0

7.13 del CFA: copertura sotto il 75% compromette l'analisi.
copertura: 100%


,ticker,nome


In [7]:
uni = pd.read_csv("universo.csv")
print(f"{len(uni)} strumenti\n")
for a in uni["area"].unique():
    sub = uni[uni["area"]==a]
    print(f"{a}: {len(sub)} — famiglie: {sub['famiglia'].value_counts().to_dict()}")
print()
uni.sort_values(["area","vol_M$"], ascending=[True,False])

30 strumenti

USA — S&P 500: 16 — famiglie: {'benchmark': 15, 'screened': 1}
USA — total mkt: 1 — famiglie: {'benchmark': 1}
Europa: 7 — famiglie: {'benchmark': 7}
Cina: 6 — famiglie: {'benchmark': 6}



,ticker,nome,area,bench,famiglia,giorni,zeri_%,vol_M$,da,ter_%
28,FXI,iShares China Large-Cap ETF,Cina,MCHI,benchmark,1890,0.7,907.5,2019-01-02,73.00
29,KWEB,KraneShares CSI China Internet ETF,Cina,MCHI,benchmark,1890,0.4,388.3,2019-01-02,70.00
24,MCHI,iShares MSCI China ETF,Cina,MCHI,benchmark,1890,0.6,190.2,2019-01-02,59.00
25,ASHR,Xtrackers Harvest CSI 300 China A-Shares ETF,Cina,MCHI,benchmark,1890,1.3,120.1,2019-01-02,65.00
27,EMXC,iShares MSCI Emerging Markets ex China ETF,Cina,MCHI,benchmark,1890,1.2,22.9,2019-01-02,25.00
26,CQQQ,Invesco China Technology ETF,Cina,MCHI,benchmark,1890,0.4,10.3,2019-01-02,65.00
17,VGK,Vanguard FTSEEuropean ETF,Europa,VGK,benchmark,1890,0.7,184.2,2019-01-02,6.00
20,EZU,iShares MSCI Eurozone ETF,Europa,VGK,benchmark,1890,0.7,113.8,2019-01-02,50.00
22,IEUR,iShares Core MSCI Europe ETF,Europa,VGK,benchmark,1890,0.8,28.9,2019-01-02,10.00
18,BBEU,JPMorgan BetaBuilders Europe ETF,Europa,VGK,benchmark,1890,1.4,18.0,2019-01-02,9.00


In [8]:
def prova_ter(tickers):
    """yfinance da' l'expense ratio GIA' in percentuale. Nessuna moltiplicazione."""
    out = {}
    for t in tickers:
        try:
            info = yf.Ticker(t).info
            e = info.get("netExpenseRatio") or info.get("annualReportExpenseRatio")
            out[t] = round(float(e), 3) if e is not None else None
        except Exception:
            out[t] = None
    return out

uni = costruisci(u, min_vol_M=0.5, max_per_area=80)   # soglia abbassata
uni["ter_%"] = uni["ticker"].map(prova_ter(uni["ticker"].tolist()))

# validazione: presenza NON basta, serve plausibilita'
sosp = uni[(uni["ter_%"].isna()) | (uni["ter_%"] < 0.01) | (uni["ter_%"] > 3.0)]
print(f"\ncopertura TER: {100*uni['ter_%'].notna().mean():.0f}% | sospetti: {len(sosp)}")
for t, atteso in [("SPY", 0.09), ("IVV", 0.03), ("VGK", 0.06)]:
    v = uni.loc[uni["ticker"]==t, "ter_%"]
    if len(v): print(f"  {t}: letto {v.iloc[0]}  atteso ~{atteso}")

# la liquidita' non sparisce: diventa una colonna
uni["liquidita"] = pd.cut(uni["vol_M$"], [0, 1, 10, 100, 1e9],
                          labels=["sottile","media","alta","altissima"])
uni["spread_stimato_bps"] = (5 / uni["vol_M$"].clip(lower=0.1)).clip(upper=50).round(1)

uni.to_csv("universo.csv", index=False)
print(f"\n{len(uni)} strumenti")
print(uni.groupby(["area","famiglia"]).size())
print("\n" + "="*60)
print(uni.groupby("liquidita", observed=True).size().to_dict())
uni[uni["famiglia"]!="benchmark"][["ticker","nome","area","famiglia","vol_M$","liquidita","spread_stimato_bps","ter_%"]]

USA — S&P 500: testo 79 candidati... 38 sopravvissuti
USA — total mkt: testo 2 candidati... 1 sopravvissuti
Europa: testo 29 candidati... 12 sopravvissuti
Cina: testo 47 candidati... 13 sopravvissuti

SALVATO universo.csv — 64 strumenti
area             famiglia  
Cina             benchmark     11
                 esclusione     2
Europa           benchmark     12
USA — S&P 500    benchmark     33
                 clima          1
                 esg            1
                 religiosa      2
                 screened       1
USA — total mkt  benchmark      1
dtype: int64

copertura TER: 98% | sospetti: 1
  SPY: letto 0.095  atteso ~0.09
  IVV: letto 0.03  atteso ~0.03
  VGK: letto 0.06  atteso ~0.06

64 strumenti
area             famiglia  
Cina             benchmark     11
                 esclusione     2
Europa           benchmark     12
USA — S&P 500    benchmark     33
                 clima          1
                 esg            1
                 religiosa      2
     

,ticker,nome,area,famiglia,vol_M$,liquidita,spread_stimato_bps,ter_%
1,CATH,Global X S&P 500 Catholic Values ETF,USA — S&P 500,religiosa,1.1,media,4.5,0.29
2,EFIV,State Street SPDR S&P 500 ESG ETF,USA — S&P 500,esg,1.2,media,4.2,0.10
19,SNPE,Xtrackers S&P 500 Scored & Screened ETF,USA — S&P 500,screened,5.5,media,0.9,0.10
26,SPUS,SP Funds S&P 500 Sharia Industry Exclusions ETF,USA — S&P 500,religiosa,1.4,media,3.6,0.45
33,SPYX,State Street SPDR S&P 500 Fossil Fuel Reserves...,USA — S&P 500,clima,2.7,media,1.9,0.20
56,CXSE,WisdomTree China ex-State-Owned Enterprises Fund,Cina,esclusione,1.4,media,3.6,0.32
63,XCEM,Columbia EM Core ex-China ETF,Cina,esclusione,0.9,sottile,5.6,0.16


In [9]:
app = '''
import streamlit as st
import pandas as pd, numpy as np
import yfinance as yf

st.set_page_config(page_title="Progetto Super", layout="wide")
GB = 252

st.title("Progetto Super")
st.caption("Non dice cosa comprare. Dice cosa stai comprando.")

# ---------- dati: caricati UNA volta, poi in cache ----------
@st.cache_data
def universo():
    return pd.read_csv("universo.csv")

@st.cache_data(show_spinner="Scarico i prezzi (una volta sola)...")
def prezzi(tickers, inizio):
    g = yf.download(list(tickers), start=inizio, auto_adjust=True, progress=False)
    return g["Close"].dropna()

@st.cache_data(show_spinner="Calcolo le barre d'errore...")
def bootstrap(tickers, inizio, bench, n=800):
    px = prezzi(tickers, inizio)
    r = px.pct_change().dropna()
    rng = np.random.default_rng(42)
    n_g = len(r)
    sh = lambda s: s.mean()/s.std()*np.sqrt(GB)
    out = {}
    for t in r.columns:
        if t == bench: continue
        d = np.empty(n)
        for i in range(n):
            c = r.iloc[rng.integers(0, n_g, n_g)]
            d[i] = sh(c[t]) - sh(c[bench])
        lo, hi = np.percentile(d, [5, 95])
        out[t] = (round(sh(r[t])-sh(r[bench]),3), round(lo,3), round(hi,3), not (lo<0<hi))
    return out

def metriche(px):
    r = px.pct_change().dropna()
    anni = len(r)/GB
    righe = []
    for t in px.columns:
        s = r[t]; curva = (1+s).cumprod(); dd = curva/curva.cummax()-1
        righe.append({"ticker": t,
            "cagr_%": round(((px[t].iloc[-1]/px[t].iloc[0])**(1/anni)-1)*100, 2),
            "vol_%": round(s.std()*np.sqrt(GB)*100, 2),
            "sharpe": round(s.mean()/s.std()*np.sqrt(GB), 2),
            "max_dd_%": round(dd.min()*100, 2)})
    return pd.DataFrame(righe).set_index("ticker")

def punteggio(tab, pesi):
    """Rango, non min-max: immune agli outlier e alle scale schiacciate.
    NON dice quanto uno e' meglio. Dice in che ordine stanno, dati quei pesi."""
    c = pd.DataFrame(index=tab.index)
    n = len(tab)
    for col, w in pesi.items():
        c[col] = tab[col].astype(float).rank(method="average")/n*100*w
    tot = sum(abs(w) for w in pesi.values()) or 1
    out = tab.copy()
    out["punteggio"] = (c.sum(axis=1)/tot).round(1)
    return out.sort_values("punteggio", ascending=False)

# ---------- barra laterale ----------
uni = universo()

with st.sidebar:
    st.header("Universo")
    area = st.selectbox("Area (confronto solo dentro l'area)", sorted(uni["area"].unique()))
    sub = uni[uni["area"]==area]

    fam = st.multiselect("Famiglia", sorted(sub["famiglia"].unique()),
                         default=sorted(sub["famiglia"].unique()))
    vol_min = st.slider("Volume minimo (M$/giorno)", 0.0, 50.0, 0.5, 0.5)
    inizio = st.date_input("Da", pd.Timestamp("2019-01-02")).isoformat()

    st.header("Pesi")
    st.caption("Spostali. La classifica si riordina.")
    w_ter = st.slider("Costo (TER) — penalizza", 0, 5, 3)
    w_sha = st.slider("Rischio/rendimento (Sharpe)", 0, 5, 3)
    w_dd  = st.slider("Dolore (max drawdown)", 0, 5, 1)
    w_te  = st.slider("Distanza dal benchmark (TE)", -5, 5, 0,
                      help="Negativo: voglio replicare. Positivo: voglio scommettere.")

sel = sub[(sub["famiglia"].isin(fam)) & (sub["vol_M$"] >= vol_min)]
bench = sel["bench"].iloc[0] if len(sel) else None
if bench and bench not in sel["ticker"].values:
    sel = pd.concat([uni[uni["ticker"]==bench], sel])

if len(sel) < 2:
    st.warning(f"Solo {len(sel)} strumenti passano i filtri. Servono almeno 2.")
    st.stop()

tick = tuple(sel["ticker"])
px = prezzi(tick, inizio)
if bench not in px.columns:
    st.error(f"Benchmark {bench} senza dati nel periodo."); st.stop()

tab = metriche(px)
r = px.pct_change().dropna()
tab["te_%"]  = [round((r[t]-r[bench]).std()*np.sqrt(GB)*100, 2) for t in tab.index]
for c in ["ter_%","famiglia","vol_M$","liquidita","spread_stimato_bps","nome"]:
    tab[c] = tab.index.map(sel.set_index("ticker")[c])

pesi = {"ter_%": -w_ter, "sharpe": w_sha, "max_dd_%": w_dd, "te_%": w_te}
pesi = {k:v for k,v in pesi.items() if v != 0}
if not pesi:
    st.warning("Tutti i pesi a zero: nessun ordinamento possibile."); st.stop()

p = punteggio(tab, pesi)

st.info(f"**{area}** · benchmark **{bench}** · {r.index.min().date()} → {r.index.max().date()} "
        f"· {len(r)} giorni · {len(p)} strumenti · risk-free assunto 0% "
        f"· fonte yahoo/Close auto_adjust · normalizzazione: rango")

st.subheader("Classifica")
st.caption(f"Pesi: {pesi}")
st.dataframe(p[["punteggio","sharpe","cagr_%","vol_%","max_dd_%","ter_%","te_%",
                "famiglia","liquidita","spread_stimato_bps"]], use_container_width=True, height=420)

# ---------- barre d'errore: solo primi 12 ----------
st.subheader("Queste differenze sono distinguibili dal rumore?")
top = [t for t in p.index[:12]]
if bench not in top: top.append(bench)
bs = bootstrap(tuple(top), inizio, bench)

c1, c2 = st.columns(2)
dist = [t for t,(d,lo,hi,ok) in bs.items() if ok]
c1.metric("Distinguibili dal benchmark", f"{len(dist)} su {len(bs)}")
c2.metric("Indistinguibili (= rumore)", f"{len(bs)-len(dist)} su {len(bs)}")

for t,(d,lo,hi,ok) in sorted(bs.items(), key=lambda x: -x[1][0]):
    col = ":green" if ok else ":orange"
    txt = "distinguibile" if ok else "non distinguibile dal rumore"
    st.write(f"**{t}** ΔSharpe {d:+.3f} [{lo:+.3f}; {hi:+.3f}] — {col}[{txt}]")
st.caption("Bootstrap appaiato su giorni singoli, 800 ricampionamenti. "
           "Assume giorni indipendenti: l'intervallo vero e' piu' largo.")

# ---------- bandiere ----------
st.divider()
st.subheader("Cosa monitorare")
st.caption("Non toccano il punteggio. Una bandiera che non si accende e' informazione anche lei.")
for t in p.index:
    f = []
    te, ter, sp = p.loc[t,"te_%"], p.loc[t,"ter_%"], p.loc[t,"spread_stimato_bps"]
    if t != bench and te < 1.0:
        f.append(f"TE {te:.2f}% — indistinguibile dal benchmark: paghi il filtro, compri l'indice")
    if pd.notna(sp) and pd.notna(ter) and sp > ter*50:
        f.append(f"spread stimato {sp:.1f}bps vs TER {ter:.2f}% — entrata+uscita costano ~{2*sp/100:.2f}pp: piu' di {2*sp/(ter*100):.1f} anni di TER")
    if p.loc[t,"liquidita"] == "sottile":
        f.append("liquidita' sottile — spread reale probabilmente peggiore della stima")
    if pd.notna(ter) and ter > p.loc[bench,"ter_%"]*3:
        f.append(f"TER {ter:.2f}% contro {p.loc[bench,'ter_%']:.2f}% del benchmark — {ter/p.loc[bench,'ter_%']:.0f}x")
    if f:
        st.write(f"**{t}** — {p.loc[t,'nome'][:50]}")
        for x in f: st.write(f"   - {x}")

st.divider()
st.caption("Le performance passate non predicono quelle future. Nessun numero qui e' una raccomandazione.")
'''

with open("app.py", "w") as fh:
    fh.write(app)
print("app.py riscritta —", len(app.splitlines()), "righe")

app.py riscritta — 160 righe


In [10]:
import numpy as np, pandas as pd, yfinance as yf
GB = 252

def calcola_bootstrap(uni, inizio="2019-01-02", n=500):
    """Gira UNA volta nel notebook. Salva su CSV. L'app legge il CSV."""
    righe = []
    for area in uni["area"].unique():
        sub = uni[uni["area"]==area]
        bench = sub["bench"].iloc[0]
        tick = sorted(set(sub["ticker"]) | {bench})
        px = yf.download(tick, start=inizio, auto_adjust=True, progress=False)["Close"].dropna()
        if bench not in px.columns or len(px) < 300:
            print(f"{area}: saltata"); continue

        r = px.pct_change().dropna()
        A = r.to_numpy(); ng, _ = A.shape
        ib = list(r.columns).index(bench)
        rng = np.random.default_rng(42)

        # a blocchi da 100 per non saturare la memoria
        D = []
        for start in range(0, n, 100):
            k = min(100, n-start)
            S = A[rng.integers(0, ng, size=(k, ng))]
            sh = S.mean(1) / S.std(1, ddof=0) * np.sqrt(GB)
            D.append(sh - sh[:, [ib]])
        D = np.vstack(D)

        sh0 = A.mean(0) / A.std(0, ddof=0) * np.sqrt(GB)
        for j, t in enumerate(r.columns):
            if t == bench: continue
            lo, hi = np.percentile(D[:, j], [5, 95])
            righe.append({"area": area, "ticker": t, "bench": bench,
                          "d_sharpe": round(float(sh0[j]-sh0[ib]), 3),
                          "ic90_lo": round(float(lo), 3), "ic90_hi": round(float(hi), 3),
                          "distinguibile": bool(not (lo < 0 < hi))})
        print(f"{area}: {len(r.columns)-1} confronti, {len(r)} giorni")

    out = pd.DataFrame(righe)
    out.to_csv("bootstrap.csv", index=False)
    print(f"\nSALVATO bootstrap.csv — {len(out)} righe")
    print(f"distinguibili: {out['distinguibile'].sum()} su {len(out)}")
    return out

uni = pd.read_csv("universo.csv")
bs = calcola_bootstrap(uni)
bs

USA — S&P 500: 37 confronti, 967 giorni
USA — total mkt: 0 confronti, 1889 giorni
Europa: 11 confronti, 1889 giorni
Cina: 12 confronti, 1889 giorni

SALVATO bootstrap.csv — 60 righe
distinguibili: 6 su 60


,area,ticker,bench,d_sharpe,ic90_lo,ic90_hi,distinguibile
0,USA — S&P 500,CATH,IVV,-0.094,-0.189,-0.007,True
1,USA — S&P 500,EFIV,IVV,0.002,-0.123,0.105,False
2,USA — S&P 500,IVE,IVV,-0.089,-0.498,0.292,False
3,USA — S&P 500,IVW,IVV,-0.089,-0.309,0.130,False
4,USA — S&P 500,NOBL,IVV,-0.529,-1.174,0.100,False
5,USA — S&P 500,RPG,IVV,-0.336,-0.728,0.107,False
6,USA — S&P 500,RPV,IVV,-0.352,-1.001,0.314,False
7,USA — S&P 500,RSP,IVV,-0.319,-0.705,0.068,False
8,USA — S&P 500,RSPD,IVV,-0.587,-1.147,-0.007,True
9,USA — S&P 500,RSPF,IVV,-0.486,-1.098,0.149,False


In [11]:
app = '''
import streamlit as st
import pandas as pd, numpy as np, yfinance as yf

st.set_page_config(page_title="Progetto Super", layout="wide")
GB = 252
st.title("Progetto Super")
st.caption("Non dice cosa comprare. Dice cosa stai comprando.")

@st.cache_data
def carica():
    return pd.read_csv("universo.csv"), pd.read_csv("bootstrap.csv")

@st.cache_data(show_spinner="Scarico i prezzi...")
def prezzi(tickers, inizio):
    g = yf.download(list(tickers), start=inizio, auto_adjust=True, progress=False)
    return g["Close"].dropna()

uni, boot = carica()

with st.sidebar:
    st.header("Universo")
    area = st.selectbox("Area", sorted(uni["area"].unique()))
    sub = uni[uni["area"]==area]
    fam = st.multiselect("Famiglia", sorted(sub["famiglia"].unique()),
                         default=sorted(sub["famiglia"].unique()))
    vmin = st.slider("Volume minimo (M$/g)", 0.0, 50.0, 0.5, 0.5)
    st.header("Pesi")
    st.caption("Spostali. La classifica si riordina.")
    w_ter = st.slider("Costo (TER) — penalizza", 0, 5, 3)
    w_sha = st.slider("Rischio/rendimento (Sharpe)", 0, 5, 3)
    w_dd  = st.slider("Dolore (max drawdown)", 0, 5, 1)
    w_te  = st.slider("Distanza dal benchmark (TE)", -5, 5, 0,
                      help="Negativo: voglio replicare. Positivo: voglio scommettere.")

sel = sub[sub["famiglia"].isin(fam) & (sub["vol_M$"] >= vmin)]
bench = sub["bench"].iloc[0]
if bench not in sel["ticker"].values:
    sel = pd.concat([sub[sub["ticker"]==bench], sel])
if len(sel) < 2:
    st.warning(f"Solo {len(sel)} strumenti passano i filtri."); st.stop()

px = prezzi(tuple(sorted(sel["ticker"])), "2019-01-02")
r = px.pct_change().dropna()
anni = len(r)/GB

righe = []
for t in px.columns:
    s = r[t]; curva = (1+s).cumprod(); dd = curva/curva.cummax()-1
    righe.append({"ticker": t,
        "sharpe": round(s.mean()/s.std()*np.sqrt(GB),2),
        "cagr_%": round(((px[t].iloc[-1]/px[t].iloc[0])**(1/anni)-1)*100,2),
        "vol_%": round(s.std()*np.sqrt(GB)*100,2),
        "max_dd_%": round(dd.min()*100,2),
        "te_%": round((r[t]-r[bench]).std()*np.sqrt(GB)*100,2)})
tab = pd.DataFrame(righe).set_index("ticker")
for c in ["ter_%","famiglia","liquidita","spread_stimato_bps","nome"]:
    tab[c] = tab.index.map(sel.set_index("ticker")[c])

pesi = {k:v for k,v in {"ter_%":-w_ter,"sharpe":w_sha,"max_dd_%":w_dd,"te_%":w_te}.items() if v}
if not pesi:
    st.warning("Tutti i pesi a zero: nessun ordinamento."); st.stop()

c = pd.DataFrame(index=tab.index)
for col, w in pesi.items():
    c[col] = tab[col].astype(float).rank()/len(tab)*100*w
tab["punteggio"] = (c.sum(1)/sum(abs(v) for v in pesi.values())).round(1)
p = tab.sort_values("punteggio", ascending=False)

st.info(f"**{area}** · benchmark **{bench}** · {r.index.min().date()} → {r.index.max().date()} "
        f"· {len(r)} giorni · risk-free 0% · yahoo/Close auto_adjust · rango")
st.subheader("Classifica")
st.caption(f"Pesi: {pesi}")
st.dataframe(p[["punteggio","sharpe","cagr_%","vol_%","max_dd_%","ter_%","te_%",
                "famiglia","liquidita","spread_stimato_bps"]], use_container_width=True, height=400)

st.subheader("Queste differenze sono distinguibili dal rumore?")
b = boot[(boot["area"]==area) & (boot["ticker"].isin(p.index))]
if len(b):
    c1, c2 = st.columns(2)
    c1.metric("Distinguibili", f"{b['distinguibile'].sum()} su {len(b)}")
    c2.metric("Indistinguibili (= rumore)", f"{(~b['distinguibile']).sum()} su {len(b)}")
    for _, row in b.sort_values("d_sharpe", ascending=False).iterrows():
        col = ":green" if row["distinguibile"] else ":orange"
        txt = "distinguibile" if row["distinguibile"] else "non distinguibile dal rumore"
        st.write(f"**{row['ticker']}** ΔSharpe {row['d_sharpe']:+.3f} "
                 f"[{row['ic90_lo']:+.3f}; {row['ic90_hi']:+.3f}] — {col}[{txt}]")
st.caption("Bootstrap appaiato, 500 ricampionamenti, calcolato offline su 2019-oggi. "
           "Assume giorni indipendenti: l'intervallo vero e' piu' largo.")

st.divider()
st.subheader("Cosa monitorare")
for t in p.index:
    f = []
    te, ter, sp = p.loc[t,"te_%"], p.loc[t,"ter_%"], p.loc[t,"spread_stimato_bps"]
    if t != bench and te < 1.0:
        f.append(f"TE {te:.2f}% — paghi il filtro, compri l'indice")
    if pd.notna(sp) and pd.notna(ter) and ter > 0 and sp > ter*50:
        f.append(f"spread ~{sp:.1f}bps vs TER {ter:.2f}% — entrata+uscita costano piu' di {2*sp/(ter*100):.1f} anni di TER")
    if p.loc[t,"liquidita"] == "sottile":
        f.append("liquidita' sottile — spread reale peggiore della stima")
    if f:
        st.write(f"**{t}** — {str(p.loc[t,'nome'])[:50]}")
        for x in f: st.write(f"   - {x}")

st.divider()
st.caption("Le performance passate non predicono quelle future. Le decisioni restano di Carlo.")
'''
with open("app.py","w") as fh: fh.write(app)
print("app.py riscritta — nessun calcolo pesante dentro")

app.py riscritta — nessun calcolo pesante dentro
